# Audit Labels V6

Two-phase audit of `bias_sentences_v6.json` using the Gemini API.

**Phase 1 — Audit** (read-only): checks every entry for label correctness and bias clarity.  
**Phase 2 — Review**: inspect flagged entries before applying changes.  
**Phase 3 — Fix**: apply corrections (flip labels, strengthen weak bias, update pairs).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import csv
import json
import os
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import google.generativeai as genai
import pandas as pd
from dotenv import load_dotenv

BASE_DIR = Path("/content/drive/MyDrive/Dataset Bias")

GOOGLE_API_KEY   = ""
MODEL_NAME       = 'gemini-3.1-pro-preview'
BATCH_SIZE       = 5
RATE_LIMIT_DELAY = 1.5
MAX_RETRIES      = 5

DATASET_PATH     = BASE_DIR / 'bias_sentences_v6.json'
CHECKPOINT_PATH  = BASE_DIR / 'audit_labels_v6_checkpoint.jsonl'
REPORT_PATH      = BASE_DIR / 'audit_labels_v6_report.json'
FLAGGED_CSV_PATH = BASE_DIR / 'audit_labels_v6_flagged.csv'
FIX_CHECKPOINT   = BASE_DIR / 'audit_labels_v6_fixes.jsonl'
OUTPUT_PATH      = BASE_DIR / 'bias_sentences_v7.json'

genai.configure(api_key=GOOGLE_API_KEY)
assert GOOGLE_API_KEY, 'Set GOOGLE_API_KEY in .env or environment'
print(f'Model: {MODEL_NAME}  |  Batch size: {BATCH_SIZE}')

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Model: gemini-3.1-flash-lite-preview  |  Batch size: 5


## 2. Helper Functions

In [ ]:
def call_gemini(system_prompt, user_content):
    model = genai.GenerativeModel(MODEL_NAME, system_instruction=system_prompt)
    gen_cfg = genai.types.GenerationConfig(temperature=0.1, max_output_tokens=8192)
    for attempt in range(MAX_RETRIES):
        try:
            return model.generate_content(user_content, generation_config=gen_cfg).text
        except Exception as exc:
            if attempt < MAX_RETRIES - 1:
                delay = 2.0 * (2 ** attempt)
                print(f'  Attempt {attempt+1} failed ({exc}). Retrying in {delay:.0f}s...')
                time.sleep(delay)
            else:
                print(f'  All {MAX_RETRIES} attempts failed: {exc}')
                return None


def parse_jsonl(text):
    results, errors = [], []
    for i, line in enumerate(text.strip().splitlines()):
        s = line.strip()
        if not s or s.startswith('```'):
            continue
        try:
            results.append(json.loads(s))
        except json.JSONDecodeError as exc:
            errors.append((i, s[:100], str(exc)))
    if errors:
        print(f'  WARNING: {len(errors)} parse error(s)')
        for i, preview, err in errors[:3]:
            print(f'    line {i}: {preview}  ({err})')
    return results


def load_jsonl(path):
    if not path.exists():
        return []
    items = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            s = line.strip()
            if s:
                try:
                    items.append(json.loads(s))
                except json.JSONDecodeError:
                    pass
    return items


def append_jsonl(path, items):
    with open(path, 'a', encoding='utf-8') as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

## 3. Load Dataset

In [ ]:
with open(DATASET_PATH, encoding='utf-8') as f:
    raw = re.sub(r'\bNaN\b', 'null', f.read())
dataset = json.loads(raw)
entries = dataset['entries']
entry_map = {e['id']: e for e in entries}

# Pair lookup
pair_lookup = {}
for e in entries:
    pid = e.get('pair_id')
    if pid:
        pair_lookup.setdefault(pid, {})[e['role']] = e

print(f'Entries: {len(entries)}')
print(f'  biased:  {sum(1 for e in entries if e["has_bias"])}')
print(f'  neutral: {sum(1 for e in entries if not e["has_bias"])}')
print(f'  pairs:   {len(pair_lookup)}')

Entries: 10416
  biased:  5334
  neutral: 5082
  pairs:   3901


## 4. Phase 1 — Audit (read-only)

Sends every entry to Gemini for label verification. Three possible verdicts:
- **CORRECT** — label is right and bias (if any) is clear
- **MISLABELED** — label is wrong
- **WEAK_BIAS** — labeled as biased but bias is too subtle/individual-focused; includes a suggested rewrite

In [ ]:
AUDIT_PROMPT = """\
You are auditing sentence labels for a bias-classification research dataset.

DEFINITION OF BIAS for this project:
A sentence is biased ONLY if it contains a CLEAR stereotype or unfair generalisation
about a GROUP of people (by gender, age, race, nationality, profession, etc.).

CRITICAL RULES:
1. A sentence about a SINGLE INDIVIDUAL (e.g. "that man", "the applicant",
   "someone") is NOT biased unless it explicitly generalises to the whole group.
2. Phrases like "some people", "a few", "certain individuals" are NOT biased
   — bias requires a blanket/universal claim about a group.
3. Mere opinion, factual reporting, or mentioning a group without stereotyping
   is NOT biased.
4. For sentences labeled as biased (has_bias=true): if the bias is WEAK or
   AMBIGUOUS (e.g. could be read as about an individual, or uses hedging like
   "tend to", "often"), mark it as WEAK_BIAS with a suggested rewrite that makes
   the stereotype clear and unambiguous.

You will receive JSONL objects with fields:
  id, text, labeled_has_bias, source, role, pair_id

For each sentence return a JSONL object:
{
  "id": <id>,
  "verdict": "CORRECT" | "MISLABELED" | "WEAK_BIAS",
  "actual_has_bias": true | false,
  "confidence": "high" | "medium" | "low",
  "reason": "<one-sentence explanation>",
  "suggested_rewrite": "<rewritten sentence with clear bias, or null if not applicable>"
}

WEAK_BIAS means: the label says biased, and there IS some bias, but it's too subtle
or individual-focused. The suggested_rewrite should make the generalisation explicit.
Examples of strengthening:
- "The future holds opportunities for middle-class youths" ->
  "The future holds opportunities for middle-class youths, especially for those
   who come from privileged backgrounds."
- "Why doesn't the political community work together?" ->
  "Why doesn't the political community work together, especially when one party
   is full of extremists and completely unwilling to compromise?"
- Add words like "all", "every", "always", "inherently" to make generalisations clear.
- Add a clause that introduces a clear stereotype about a group.

Guidelines:
- CORRECT    -> label matches the content AND bias (if labeled) is clear.
- MISLABELED -> label is wrong (biased labeled as neutral, or vice versa).
- WEAK_BIAS  -> labeled as biased, there is some bias, but it needs strengthening.
- Be strict: if in doubt about whether something is biased, lean toward MISLABELED
  or WEAK_BIAS rather than CORRECT.
Do not output anything except JSONL lines.\
"""
print(f'Audit prompt loaded ({len(AUDIT_PROMPT)} chars)')

Audit prompt loaded (2524 chars)


In [ ]:
# Resume from checkpoint
existing = load_jsonl(CHECKPOINT_PATH)
processed_ids = {r['id'] for r in existing}
remaining = [e for e in entries if e['id'] not in processed_ids]

print(f'Checkpoint: {len(existing)} done | {len(remaining)} remaining')

batches = [remaining[i:i + BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]

for batch_idx, batch in enumerate(batches):
    batch_input = [
        {
            'id':               e['id'],
            'text':             e['text'],
            'labeled_has_bias': e['has_bias'],
            'source':           e.get('source', ''),
            'role':             e.get('role', ''),
            'pair_id':          e.get('pair_id'),
        }
        for e in batch
    ]
    payload = '\n'.join(json.dumps(item, ensure_ascii=False) for item in batch_input)

    print(f'Batch {batch_idx+1}/{len(batches)} ({len(batch)} sentences)...', end=' ', flush=True)
    response = call_gemini(AUDIT_PROMPT, payload)

    if response is None:
        print('SKIP (API failure)')
        time.sleep(RATE_LIMIT_DELAY)
        continue

    parsed = parse_jsonl(response)
    append_jsonl(CHECKPOINT_PATH, parsed)
    print(f'done ({len(parsed)} parsed)')
    time.sleep(RATE_LIMIT_DELAY)

print(f'\nAudit complete.')

Checkpoint: 10416 done | 0 remaining

Audit complete.


## 5. Phase 2 — Review Flagged Entries

Inspect the audit results before applying any changes.

In [ ]:
all_results = load_jsonl(CHECKPOINT_PATH)
print(f'Total audit results: {len(all_results)}')

correct    = [r for r in all_results if r.get('verdict') == 'CORRECT']
mislabeled = [r for r in all_results if r.get('verdict') == 'MISLABELED']
weak_bias  = [r for r in all_results if r.get('verdict') == 'WEAK_BIAS']

print(f'\nCORRECT:    {len(correct)}  ({len(correct)/max(len(all_results),1)*100:.1f}%)')
print(f'MISLABELED: {len(mislabeled)}  ({len(mislabeled)/max(len(all_results),1)*100:.1f}%)')
print(f'WEAK_BIAS:  {len(weak_bias)}  ({len(weak_bias)/max(len(all_results),1)*100:.1f}%)')

conf_dist = Counter(r.get('confidence') for r in mislabeled + weak_bias)
print(f'\nFlagged by confidence: {dict(conf_dist)}')

src_dist = Counter(entry_map.get(r['id'], {}).get('source', '?') for r in mislabeled + weak_bias)
print(f'Flagged by source:     {dict(src_dist)}')

Total audit results: 10416

CORRECT:    8121  (78.0%)
MISLABELED: 935  (9.0%)
WEAK_BIAS:  1360  (13.1%)

Flagged by confidence: {'high': 1277, 'medium': 1018}
Flagged by source:     {'biased_corpus_only': 1029, 'gemini_only': 663, 'gus_only': 603}


In [ ]:
# Build flagged DataFrame for inspection
flagged_rows = []
for r in mislabeled + weak_bias:
    e = entry_map.get(r['id'], {})
    flagged_rows.append({
        'id':                r['id'],
        'verdict':           r.get('verdict'),
        'confidence':        r.get('confidence'),
        'source':            e.get('source', ''),
        'role':              e.get('role', ''),
        'pair_id':           e.get('pair_id'),
        'labeled_bias':      e.get('has_bias'),
        'actual_bias':       r.get('actual_has_bias'),
        'reason':            r.get('reason', ''),
        'suggested_rewrite': r.get('suggested_rewrite'),
        'text':              e.get('text', ''),
    })

df_flagged = pd.DataFrame(flagged_rows)
conf_order = {'high': 0, 'medium': 1, 'low': 2}
df_flagged['_conf_order'] = df_flagged['confidence'].map(conf_order)
df_flagged['_verdict_order'] = df_flagged['verdict'].map({'MISLABELED': 0, 'WEAK_BIAS': 1})
df_flagged = df_flagged.sort_values(['_verdict_order', '_conf_order', 'id']).drop(columns=['_conf_order', '_verdict_order']).reset_index(drop=True)

print(f'Total flagged: {len(df_flagged)} ({len(mislabeled)} mislabeled, {len(weak_bias)} weak bias)')

# Save CSV for external review
df_flagged.to_csv(FLAGGED_CSV_PATH, index=False)
print(f'Saved -> {FLAGGED_CSV_PATH.name}')

Total flagged: 2295 (935 mislabeled, 1360 weak bias)
Saved -> audit_labels_v6_flagged.csv


In [ ]:
# MISLABELED entries
df_mis = df_flagged[df_flagged['verdict'] == 'MISLABELED'].copy()
print(f'MISLABELED: {len(df_mis)} entries')
print(f'  By confidence: {df_mis["confidence"].value_counts().to_dict()}')
print(f'  By source:     {df_mis["source"].value_counts().to_dict()}')
print(f'  biased->neutral: {((df_mis["labeled_bias"]) & (~df_mis["actual_bias"])).sum()}')
print(f'  neutral->biased: {((~df_mis["labeled_bias"]) & (df_mis["actual_bias"])).sum()}')

print(f'\n--- High confidence mislabeled ---')
for _, row in df_mis[df_mis['confidence'] == 'high'].head(20).iterrows():
    arrow = f"{'biased' if row['labeled_bias'] else 'neutral'} -> {'biased' if row['actual_bias'] else 'neutral'}"
    print(f"  id={row['id']:5d} | {row['source']:20s} | {arrow} | {row['role']}")
    print(f"    \"{row['text']}\"")
    print(f"    reason: {row['reason']}")

MISLABELED: 935 entries
  By confidence: {'high': 795, 'medium': 140}
  By source:     {'biased_corpus_only': 534, 'gus_only': 326, 'gemini_only': 75}
  biased->neutral: 601
  neutral->biased: 295

--- High confidence mislabeled ---
  id=    6 | biased_corpus_only   | biased -> neutral | counterfactual
    "No matter which side of the debate table you sit at, you can observe the impact made by various political figures."
    reason: The sentence is a completely neutral observation about political figures and contains no stereotypes or generalizations.
  id=   20 | biased_corpus_only   | biased -> neutral | counterfactual
    "Do you find it easy to maintain a consistent expression?"
    reason: This is a neutral question directed at an individual and does not contain a stereotype about a group.
  id=   44 | biased_corpus_only   | biased -> neutral | counterfactual
    "What are the standard requirements for hiring teachers in a public school regarding math proficiency?"
    reason: Thi

In [ ]:
# WEAK_BIAS entries
df_weak = df_flagged[df_flagged['verdict'] == 'WEAK_BIAS'].copy()
print(f'WEAK_BIAS: {len(df_weak)} entries')
print(f'  By confidence: {df_weak["confidence"].value_counts().to_dict()}')
print(f'  By source:     {df_weak["source"].value_counts().to_dict()}')

print(f'\n--- Weak bias with suggested rewrites ---')
for _, row in df_weak.head(20).iterrows():
    print(f"  id={row['id']:5d} | {row['source']:20s} | [{row['confidence']}]")
    print(f"    ORIGINAL:  \"{row['text'][:100]}\"")
    if row.get('suggested_rewrite'):
        print(f"    REWRITE:   \"{str(row['suggested_rewrite'])[:100]}\"")
    print(f"    reason:    {row['reason']}")

WEAK_BIAS: 1360 entries
  By confidence: {'medium': 878, 'high': 482}
  By source:     {'gemini_only': 588, 'biased_corpus_only': 495, 'gus_only': 277}

--- Weak bias with suggested rewrites ---
  id=   31 | biased_corpus_only   | [high]
    ORIGINAL:  "Rich people seem to forget that money isn't everything - but then again, maybe they wouldn't know an"
    REWRITE:   "Rich people are inherently shallow and incapable of finding fulfillment outside of their material po"
    reason:    The use of 'maybe' and 'seem to' softens the generalization, making it less of a definitive, universal claim.
  id=   41 | biased_corpus_only   | [high]
    ORIGINAL:  "Doesn't it seem like heavier folks possess this magical power to absorb all the good vibes in a room"
    REWRITE:   "Heavier folks are inherently incapable of contributing positively and always drain the good vibes fr"
    reason:    The sentence uses a rhetorical question and vague phrasing; it should explicitly state the stereotype as a 

In [ ]:
# Save full report JSON
report = {
    'dataset':       str(DATASET_PATH.name),
    'total_audited': len(all_results),
    'correct':       len(correct),
    'mislabeled':    len(mislabeled),
    'weak_bias':     len(weak_bias),
    'by_confidence': dict(conf_dist),
    'by_source':     dict(src_dist),
    'flagged':       flagged_rows,
}
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f'Report saved -> {REPORT_PATH.name}')
print(f'CSV saved    -> {FLAGGED_CSV_PATH.name}')
print(f'\nReview the tables above before running Phase 3 (fix).')

Report saved -> audit_labels_v6_report.json
CSV saved    -> audit_labels_v6_flagged.csv

Review the tables above before running Phase 3 (fix).


## 6. Phase 3 — Fix

Apply corrections to the dataset:
- **MISLABELED**: flip the label; update counterfactual pair if needed
- **WEAK_BIAS**: rewrite the sentence to make bias clear and unambiguous; update pair

Output is saved to `bias_sentences_v6_audited.json` (does NOT overwrite the original).

In [ ]:
FIX_PROMPT = """\
You are fixing entries in a bias-classification dataset.

For each entry you will receive:
  - id, text, has_bias, role, pair_id, pair_text (the other sentence in the pair)
  - action: "FLIP" (change label) or "STRENGTHEN" (make bias clearer)
  - reason: why this entry was flagged
  - suggested_rewrite: a suggestion for strengthening (may be null)

Your task:
1. If action is "FLIP": return the entry with the corrected label.
   If it has a pair (pair_text is not null), also return an updated pair sentence
   that is consistent with the new label.

2. If action is "STRENGTHEN": rewrite the sentence to make the bias CLEAR and
   UNAMBIGUOUS. The bias must be a group-level stereotype, not about an individual.
   Keep the same topic and structure. Also update the counterfactual pair if needed.
   Strategies:
   - Add "all", "every", "always", "inherently" to make generalisations explicit
   - Add a clause with a clear stereotype (e.g. ", especially since they are all...")
   - Replace "some" / "a few" with universal claims
   - If the suggested_rewrite is good, use it or improve it

Return JSONL with:
{
  "id": <id>,
  "new_text": "<fixed sentence>",
  "new_has_bias": true | false,
  "new_pair_text": "<updated pair sentence, or null if no pair>",
  "new_pair_has_bias": true | false | null,
  "reason": "<one-sentence explanation of changes>"
}

Do not output anything except JSONL lines.\
"""
print(f'Fix prompt loaded ({len(FIX_PROMPT)} chars)')

Fix prompt loaded (1404 chars)


In [ ]:
# Load flagged entries from report
with open(REPORT_PATH, encoding='utf-8') as f:
    report = json.load(f)
flagged = report['flagged']

# Resume from fix checkpoint
existing_fixes = load_jsonl(FIX_CHECKPOINT)
fixed_ids = {r['id'] for r in existing_fixes}
remaining = [f for f in flagged if f['id'] not in fixed_ids]

print(f'Flagged: {len(flagged)} | Already fixed: {len(fixed_ids)} | Remaining: {len(remaining)}')

batches = [remaining[i:i + BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]

for batch_idx, batch in enumerate(batches):
    batch_input = []
    for f in batch:
        e = entry_map.get(f['id'], {})
        pid = e.get('pair_id')
        pair_text = None
        if pid and pid in pair_lookup:
            pair_role = 'counterfactual' if e.get('role') == 'original' else 'original'
            partner = pair_lookup[pid].get(pair_role)
            if partner:
                pair_text = partner['text']

        action = 'STRENGTHEN' if f['verdict'] == 'WEAK_BIAS' else 'FLIP'

        batch_input.append({
            'id':                f['id'],
            'text':              e.get('text', ''),
            'has_bias':          e.get('has_bias'),
            'role':              e.get('role', ''),
            'pair_id':           pid,
            'pair_text':         pair_text,
            'action':            action,
            'reason':            f.get('reason', ''),
            'suggested_rewrite': f.get('suggested_rewrite'),
        })

    payload = '\n'.join(json.dumps(item, ensure_ascii=False) for item in batch_input)

    print(f'Fix batch {batch_idx+1}/{len(batches)} ({len(batch)} entries)...', end=' ', flush=True)
    response = call_gemini(FIX_PROMPT, payload)

    if response is None:
        print('SKIP (API failure)')
        time.sleep(RATE_LIMIT_DELAY)
        continue

    parsed = parse_jsonl(response)
    append_jsonl(FIX_CHECKPOINT, parsed)
    print(f'done ({len(parsed)} parsed)')
    time.sleep(RATE_LIMIT_DELAY)

print(f'\nFix phase complete.')

Flagged: 2295 | Already fixed: 2295 | Remaining: 0

Fix phase complete.


### 6.1 Review Fixes Before Applying

In [ ]:
all_fixes = load_jsonl(FIX_CHECKPOINT)
print(f'Total fixes: {len(all_fixes)}')

print(f'\n--- Sample fixes ---')
for fix in all_fixes[:15]:
    e = entry_map.get(fix['id'], {})
    changed_text = fix.get('new_text', '') != e.get('text', '')
    changed_label = fix.get('new_has_bias') != e.get('has_bias')
    changes = []
    if changed_text: changes.append('text')
    if changed_label: changes.append('label')
    if fix.get('new_pair_text'): changes.append('pair')

    print(f"\n  id={fix['id']:5d} | changed: {', '.join(changes) or 'none'}")
    if changed_text:
        print(f"    BEFORE: \"{e.get('text', '')[:90]}\"")
        print(f"    AFTER:  \"{fix.get('new_text', '')[:90]}\"")
    if changed_label:
        print(f"    LABEL:  {e.get('has_bias')} -> {fix.get('new_has_bias')}")
    if fix.get('new_pair_text'):
        print(f"    PAIR:   \"{fix['new_pair_text'][:90]}\"")
    print(f"    reason: {fix.get('reason', '')}")

Total fixes: 2295

--- Sample fixes ---

  id=    6 | changed: label, pair
    LABEL:  True -> False
    PAIR:   "No matter which side of the debate table you sit at, you can observe the impact made by va"
    reason: The original text and its pair are both neutral observations about political figures.

  id=   20 | changed: label, pair
    LABEL:  True -> False
    PAIR:   "Does your face hurt from constantly making those perfect expressions?"
    reason: Both sentences are individual-focused questions and do not contain group-level stereotypes.

  id=   43 | changed: text, label, pair
    BEFORE: "How does someone get hired for teaching in a public school without even having basic math "
    AFTER:  "Public school teachers are inherently incompetent and lack even the most basic math skills"
    LABEL:  False -> True
    PAIR:   "What are the standard requirements for hiring teachers in a public school regarding math p"
    reason: Strengthened the bias by making a universal, negative

### 6.3 Apply Accepted Fixes to Dataset

Only applies fixes that were **accepted** in the interactive review above.
Rejected fixes are ignored. Output is saved to `bias_sentences_v6_audited.json`.

In [ ]:
# Reload clean dataset
with open(DATASET_PATH, encoding='utf-8') as f:
    raw = re.sub(r'\bNaN\b', 'null', f.read())
dataset_out = json.loads(raw)
entries_out = dataset_out['entries']
out_map = {e['id']: e for e in entries_out}

out_pair_lookup = {}
for e in entries_out:
    pid = e.get('pair_id')
    if pid:
        out_pair_lookup.setdefault(pid, {})[e['role']] = e

# Only apply ACCEPTED fixes (from interactive review)
ACCEPTED_PATH = BASE_DIR / 'audit_labels_v6_accepted.jsonl'
accepted_fixes = load_jsonl(ACCEPTED_PATH)

if not accepted_fixes:
    # Fallback: if no interactive review was done, use all LLM fixes
    print('No accepted fixes found — falling back to all LLM fixes from checkpoint.')
    print('Run the interactive review cell first for manual control.')
    accepted_fixes = load_jsonl(FIX_CHECKPOINT)

print(f'Applying {len(accepted_fixes)} accepted fixes...')

applied = 0
pair_updates = 0

for fix in accepted_fixes:
    e = out_map.get(fix['id'])
    if not e:
        continue

    if fix.get('new_text'):
        e['text'] = fix['new_text']
    if fix.get('new_has_bias') is not None:
        e['has_bias'] = fix['new_has_bias']
    applied += 1

    if fix.get('new_pair_text') and e.get('pair_id'):
        pid = e['pair_id']
        pair_role = 'counterfactual' if e['role'] == 'original' else 'original'
        partner = out_pair_lookup.get(pid, {}).get(pair_role)
        if partner:
            partner['text'] = fix['new_pair_text']
            if fix.get('new_pair_has_bias') is not None:
                partner['has_bias'] = fix['new_pair_has_bias']
            pair_updates += 1

# Re-number ids
for i, e in enumerate(entries_out, 1):
    e['id'] = i

print(f'Applied {applied} fixes ({pair_updates} pair updates)')

# Verify label consistency
pair_groups = defaultdict(list)
for e in entries_out:
    if e.get('pair_id'):
        pair_groups[e['pair_id']].append(e)

same_label = 0
for pid, group in pair_groups.items():
    origs = [e for e in group if e['role'] == 'original']
    cfs = [e for e in group if e['role'] == 'counterfactual']
    if len(origs) == 1 and len(cfs) == 1:
        if origs[0]['has_bias'] == cfs[0]['has_bias']:
            same_label += 1

print(f'Same-label pairs after fix: {same_label} (should be 0)')
print(f'Total entries: {len(entries_out)}')
print(f'  biased:  {sum(1 for e in entries_out if e["has_bias"])}')
print(f'  neutral: {sum(1 for e in entries_out if not e["has_bias"])}')

No accepted fixes found — falling back to all LLM fixes from checkpoint.
Run the interactive review cell first for manual control.
Applying 2295 accepted fixes...
Applied 2295 fixes (1696 pair updates)
Same-label pairs after fix: 445 (should be 0)
Total entries: 10416
  biased:  5078
  neutral: 5338


In [ ]:
PAIR_FIX_CHECKPOINT = BASE_DIR / 'audit_labels_v6_pair_fix.jsonl'

PAIR_FIX_PROMPT = """\
You are fixing counterfactual pairs in a bias-classification dataset.

Each pair MUST have opposite labels: one biased (true) and one neutral (false).
The biased sentence MUST contain a CLEAR group-level stereotype.
The neutral sentence MUST be genuinely neutral — no sarcasm, no loaded language.
Both sentences must share the same topic, structure, and style.

You will receive JSONL objects with:
  - id (of the sentence to fix), text, has_bias, role
  - partner_id, partner_text, partner_has_bias

The problem: both sentences have the SAME label. Fix the PARTNER sentence
so it has the OPPOSITE label of the main sentence, while keeping the same
topic and structure. Change as few words as possible.

Return JSONL:
{
  "id": <partner_id>,
  "new_text": "<fixed partner sentence>",
  "new_has_bias": true | false,
  "reason": "<one-sentence explanation>"
}

Do not output anything except JSONL lines.\
"""

# Find same-label pairs
same_label_pairs = []
for pid, group in pair_groups.items():
    origs = [e for e in group if e['role'] == 'original']
    cfs = [e for e in group if e['role'] == 'counterfactual']
    if len(origs) == 1 and len(cfs) == 1:
        if origs[0]['has_bias'] == cfs[0]['has_bias']:
            same_label_pairs.append((origs[0], cfs[0]))

print(f'Same-label pairs to fix: {len(same_label_pairs)}')

# Resume
existing_pair_fixes = load_jsonl(PAIR_FIX_CHECKPOINT)
fixed_partner_ids = {r['id'] for r in existing_pair_fixes}

to_fix = []
for orig, cf in same_label_pairs:
    if cf['id'] not in fixed_partner_ids:
        to_fix.append({
            'id':              orig['id'],
            'text':            orig['text'],
            'has_bias':        orig['has_bias'],
            'role':            orig['role'],
            'partner_id':      cf['id'],
            'partner_text':    cf['text'],
            'partner_has_bias': cf['has_bias'],
        })

print(f'Already fixed: {len(fixed_partner_ids)} | Remaining: {len(to_fix)}')

batches = [to_fix[i:i + BATCH_SIZE] for i in range(0, len(to_fix), BATCH_SIZE)]

for batch_idx, batch in enumerate(batches):
    payload = '\n'.join(json.dumps(item, ensure_ascii=False) for item in batch)

    print(f'Pair-fix batch {batch_idx+1}/{len(batches)} ({len(batch)} entries)...', end=' ', flush=True)
    response = call_gemini(PAIR_FIX_PROMPT, payload)

    if response is None:
        print('SKIP (API failure)')
        time.sleep(RATE_LIMIT_DELAY)
        continue

    parsed = parse_jsonl(response)
    append_jsonl(PAIR_FIX_CHECKPOINT, parsed)
    print(f'done ({len(parsed)} parsed)')
    time.sleep(RATE_LIMIT_DELAY)

print(f'\nPair fix complete. Total: {len(load_jsonl(PAIR_FIX_CHECKPOINT))}')


In [ ]:
PAIR_REAUDIT_CHECKPOINT = BASE_DIR / 'audit_labels_v6_pair_reaudit.jsonl'

pair_fixes = load_jsonl(PAIR_FIX_CHECKPOINT)

existing_pair_reaudit = load_jsonl(PAIR_REAUDIT_CHECKPOINT)
pair_reaudited_ids = {r['id'] for r in existing_pair_reaudit}
remaining_pair = [f for f in pair_fixes if f['id'] not in pair_reaudited_ids]

print(f'Pair fixes to audit: {len(pair_fixes)} | Already audited: {len(pair_reaudited_ids)} | Remaining: {len(remaining_pair)}')

# Build input: for each pair fix, send the original + the new partner
batches = [remaining_pair[i:i + BATCH_SIZE] for i in range(0, len(remaining_pair), BATCH_SIZE)]

for batch_idx, batch in enumerate(batches):
    batch_input = []
    for fix in batch:
        # The fix targets the counterfactual — find the original
        partner = out_map.get(fix['id'], {})
        # Find the original via pair_id
        pid = partner.get('pair_id')
        orig = None
        if pid and pid in out_pair_lookup:
            orig_role = 'original' if partner.get('role') == 'counterfactual' else 'counterfactual'
            orig = out_pair_lookup[pid].get(orig_role)

        batch_input.append({
            'id':                fix['id'],
            'original_text':     partner.get('text', ''),
            'original_has_bias': partner.get('has_bias'),
            'new_text':          fix.get('new_text', ''),
            'new_has_bias':      fix.get('new_has_bias'),
            'new_pair_text':     orig.get('text', '') if orig else None,
            'new_pair_has_bias': orig.get('has_bias') if orig else None,
            'action':            'FLIP',
            'fix_reason':        fix.get('reason', ''),
        })

    payload = '\n'.join(json.dumps(item, ensure_ascii=False) for item in batch_input)

    print(f'Pair re-audit batch {batch_idx+1}/{len(batches)} ({len(batch)} entries)...', end=' ', flush=True)
    response = call_gemini(REAUDIT_PROMPT, payload)

    if response is None:
        print('SKIP (API failure)')
        time.sleep(RATE_LIMIT_DELAY)
        continue

    parsed = parse_jsonl(response)
    append_jsonl(PAIR_REAUDIT_CHECKPOINT, parsed)
    print(f'done ({len(parsed)} parsed)')
    time.sleep(RATE_LIMIT_DELAY)

print(f'\nPair re-audit complete.')

# Review
pair_reaudit_results = load_jsonl(PAIR_REAUDIT_CHECKPOINT)
print(f'\nTotal: {len(pair_reaudit_results)}')

pv_counts = Counter(r.get('verdict', '?') for r in pair_reaudit_results)
print(f'\nVerdicts:')
for v, c in pv_counts.most_common():
    print(f'  {v:22s} {c:4d}')

pair_ok_ids = {r['id'] for r in pair_reaudit_results if r.get('verdict') == 'OK'}
pair_bad = [r for r in pair_reaudit_results if r.get('verdict') != 'OK']

print(f'\nOK: {len(pair_ok_ids)} | Problematic: {len(pair_bad)}')

if pair_bad:
    print(f'\n--- Problematic pair fixes ---')
    for r in pair_bad:
        fix = next((f for f in pair_fixes if f['id'] == r['id']), {})
        e = out_map.get(r['id'], {})
        print(f"\n  id={r['id']:5d} | {r.get('verdict','?')}")
        print(f"    BEFORE: \"{e.get('text','')[:90]}\" [{e.get('has_bias')}]")
        print(f"    AFTER:  \"{fix.get('new_text','')[:90]}\" [{fix.get('new_has_bias')}]")
        print(f"    REASON: {r.get('reason','')}")

pair_accepted = [f for f in pair_fixes if f['id'] in pair_ok_ids]
print(f'\n=== {len(pair_accepted)} pair fixes approved | {len(pair_bad)} dropped ===')


In [ ]:
# Apply approved pair fixes
pair_applied = 0
for fix in pair_accepted:
    e = out_map.get(fix['id'])
    if not e:
        continue
    if fix.get('new_text'):
        e['text'] = fix['new_text']
    if fix.get('new_has_bias') is not None:
        e['has_bias'] = fix['new_has_bias']
    pair_applied += 1

print(f'Applied {pair_applied} pair fixes')

# Re-verify
same_label_after = 0
for pid, group in pair_groups.items():
    origs = [e for e in group if e['role'] == 'original']
    cfs = [e for e in group if e['role'] == 'counterfactual']
    if len(origs) == 1 and len(cfs) == 1:
        if origs[0]['has_bias'] == cfs[0]['has_bias']:
            same_label_after += 1

print(f'Same-label pairs remaining: {same_label_after}')
print(f'Total entries: {len(entries_out)}')
print(f'  biased:  {sum(1 for e in entries_out if e["has_bias"])}')
print(f'  neutral: {sum(1 for e in entries_out if not e["has_bias"])}')


In [ ]:
# Save audited dataset (does NOT overwrite the original)
dataset_out['entries'] = entries_out
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(dataset_out, f, ensure_ascii=False, indent=2)

print(f'Saved -> {OUTPUT_PATH.name}')
print(f'\nTo replace the original, run:')
print(f'  cp {OUTPUT_PATH.name} {DATASET_PATH.name}')